In [2]:
import networkx as nx
from pathlib import Path
from dash import Dash, html, dcc
import dash_cytoscape as cyto
from dash.dependencies import Input, Output, State, ALL
from dash.exceptions import PreventUpdate
import dash

NODE_TYPE_ATTR = "Node_Type"

# Node types used for show/hide buttons
NODE_TYPES = [
    "concept",
    "people",
    "location",
    "conjunction",
    "relationship",
    "target",
    "other"
]

# Paths to our GraphML files
# file1 = Path(r"C:\WorldView\worldview\w0rldview\test1.graphml")
# file2 = Path(r"C:\WorldView\worldview\w0rldview\test2.graphml")

# Mac
file1 = Path("/Users/nickpellegri/Documents/Github/w0rldview/test1.graphml")
file2 = Path("/Users/nickpellegri/Documents/Github/w0rldview/test2.graphml")


# Function to load a GraphML file and convert it to Cytoscape elements
def load_and_make_elements(path, seed=42):
    G = nx.read_graphml(path)
    pos = nx.spring_layout(G, seed=seed)

    scale = 700
    offset = 350

    def to_pos(p):
        return {
            "x": float(p[0] * scale + offset),
            "y": float(p[1] * scale + offset)
        }

    nodes = []
    for n in G.nodes():
        node_type = str(G.nodes[n].get(NODE_TYPE_ATTR, "other")).strip().lower()

        nodes.append({
            "data": {
                "id": str(n),
                "label": str(n),
                "type": node_type
            },
            "position": to_pos(pos[n]),
        })

    edges = []
    for u, v in G.edges():
        edges.append({
            "data": {
                "id": f"{u}__{v}",
                "source": str(u),
                "target": str(v)
            }
        })

    return G, nodes, edges


# Load both graphs and create Cytoscape elements
G1, nodes1, edges1 = load_and_make_elements(file1, seed=42)
G2, nodes2, edges2 = load_and_make_elements(file2, seed=7)

matching_nodes = set(G1.nodes()) & set(G2.nodes())
matching_edges = set(G1.edges()) & set(G2.edges())


def mark_elements(nodes, edges, matching_nodes, matching_edges):
    out = []
    undirected = {(b, a) for (a, b) in matching_edges}

    for n in nodes:
        n_id = n["data"]["id"]
        n["classes"] = "match" if n_id in matching_nodes else "unique"
        out.append(n)

    for e in edges:
        s = e["data"]["source"]
        t = e["data"]["target"]
        cls = "match" if (s, t) in matching_edges or (s, t) in undirected else "unique"
        e["classes"] = cls
        out.append(e)

    return out


elements1_original = mark_elements(nodes1, edges1, matching_nodes, matching_edges)
elements2_original = mark_elements(nodes2, edges2, matching_nodes, matching_edges)


# Cytoscape stylesheet for node types and match/unique styling
stylesheet = [
    {
        "selector": "node",
        "style": {
            "label": "data(label)",
            "width": 60,
            "height": 60,
            "font-size": 10,
            "text-valign": "center",
            "text-halign": "center",
            "border-color": "#222",
            "border-width": 1,
        }
    },
    {
        "selector": "edge",
        "style": {
            "width": 3,
            "line-color": "#bbb",
            "target-arrow-color": "#bbb",
            "target-arrow-shape": "triangle",
            "curve-style": "bezier"
        }
    },

    # Node type colors and shapes
    {
        "selector": 'node[type = "concept"]',
        "style": {
            "background-color": "purple",
            "shape": "diamond"
        }
    },
    {
        "selector": 'node[type = "people"]',
        "style": {
            "background-color": "blue",
            "shape": "ellipse"
        }
    },
    {
        "selector": 'node[type = "location"]',
        "style": {
            "background-color": "green",
            "shape": "rectangle"
        }
    },
    {
        "selector": 'node[type = "conjunction"]',
        "style": {
            "background-color": "yellow",
            "shape": "triangle"
        }
    },
    {
        "selector": 'node[type = "relationship"]',
        "style": {
            "background-color": "orange",
            "shape": "hexagon"
        }
    },
    {
        "selector": 'node[type = "target"]',
        "style": {
            "background-color": "red",
            "shape": "star"
        }
    },
    {
        "selector": 'node[type = "other"]',
        "style": {
            "background-color": "gray",
            "shape": "ellipse"
        }
    },

    # Match and unique styles
    {
        "selector": "node.match",
        "style": {
            "opacity": 0.9,
            "border-width": 1,
            "border-color": "#444"
        }
    },
    {
        "selector": "node.unique",
        "style": {
            "opacity": 1.0,
            "border-width": 3,
            "border-color": "#b30000"
        }
    },
    {
        "selector": "edge.match",
        "style": {
            "line-color": "#92adda",
            "target-arrow-color": "#92adda",
            "opacity": 0.7
        }
    },
    {
        "selector": "edge.unique",
        "style": {
            "line-color": "#ff6b6b",
            "target-arrow-color": "#ff6b6b",
            "line-style": "dashed",
            "opacity": 1.0
        }
    },

    # Hidden styling/Opacity of Nodes and edges when "hidden" class is applied
    {
        "selector": "node.hidden",
        "style": {
            "opacity": 0.04,
            "text-opacity": 0
        }
    },
    {
        "selector": "edge.hidden",
        "style": {
            "opacity": 0.04
        }
    },
]


def toggle_hidden_class(elements, hidden_ids):
    updated = []

    for el in elements:
        data = el.get("data", {})
        classes = el.get("classes", "").split()
        classes = [c for c in classes if c != "hidden"]

        if "source" not in data:
            if data.get("id") in hidden_ids:
                classes.append("hidden")
        else:
            if data.get("source") in hidden_ids or data.get("target") in hidden_ids:
                classes.append("hidden")

        new_el = el.copy()
        new_el["classes"] = " ".join(classes)
        updated.append(new_el)

    return updated


def get_node_ids_by_type(elements, node_type):
    node_ids = []

    for el in elements:
        data = el.get("data", {})

        # This checks nodes only, not edges
        if "source" not in data:
            if data.get("type") == node_type:
                node_ids.append(data.get("id"))

    return node_ids


def make_type_buttons():
    buttons = []

    for node_type in NODE_TYPES:
        label = node_type.capitalize()

        buttons.append(
            html.Button(
                f"Show/Hide {label}",
                id={"type": "toggle-type-btn", "node_type": node_type},
                n_clicks=0,
                style={
                    "padding": "6px 10px",
                    "margin": "4px",
                    "fontSize": "13px",
                    "cursor": "pointer"
                }
            )
        )

    return buttons


app = Dash(__name__)

app.layout = html.Div([

    dcc.Store(id="hidden-store", data={
        "cytoscape-1": [],
        "cytoscape-2": []
    }),

    html.Div([
        html.Button(
            "Show All Nodes",
            id="show-all-btn",
            n_clicks=0,
            style={
                "padding": "8px 14px",
                "margin": "8px",
                "fontSize": "14px",
                "cursor": "pointer"
            }
        ),

        html.Div(
            "Click a node to hide/show it. Use the buttons below to hide/show full node types.",
            style={
                "display": "inline-block",
                "marginLeft": "10px",
                "fontSize": "14px"
            }
        )
    ]),

    html.Div(
        make_type_buttons(),
        style={
            "padding": "8px",
            "marginLeft": "8px",
            "marginBottom": "8px",
            "border": "1px solid #ddd",
            "borderRadius": "6px",
            "display": "inline-block"
        }
    ),

    html.Div([

        html.Div([
            html.H3(file1.name, style={"textAlign": "center", "margin": "6px"}),

            cyto.Cytoscape(
                id="cytoscape-1",
                elements=elements1_original,
                stylesheet=stylesheet,
                style={
                    "width": "48vw",
                    "height": "80vh",
                    "display": "inline-block"
                },
                layout={"name": "preset"},
                minZoom=0.6,
                maxZoom=2.0,
                boxSelectionEnabled=False,
                userPanningEnabled=True,
                userZoomingEnabled=True,
                autoungrabify=False,
            )
        ], style={
            "display": "inline-block",
            "verticalAlign": "top",
            "width": "49%"
        }),

        html.Div([
            html.H3(file2.name, style={"textAlign": "center", "margin": "6px"}),

            cyto.Cytoscape(
                id="cytoscape-2",
                elements=elements2_original,
                stylesheet=stylesheet,
                style={
                    "width": "48vw",
                    "height": "80vh",
                    "display": "inline-block"
                },
                layout={"name": "preset"},
                minZoom=0.6,
                maxZoom=2.0,
                boxSelectionEnabled=False,
                userPanningEnabled=True,
                userZoomingEnabled=True,
                autoungrabify=False,
            )
        ], style={
            "display": "inline-block",
            "verticalAlign": "top",
            "width": "49%"
        }),

    ], style={
        "display": "flex",
        "justifyContent": "space-between",
        "padding": "8px"
    })
])


@app.callback(
    Output("hidden-store", "data"),
    Output("cytoscape-1", "elements"),
    Output("cytoscape-2", "elements"),

    Input("cytoscape-1", "tapNodeData"),
    Input("cytoscape-2", "tapNodeData"),
    Input("show-all-btn", "n_clicks"),
    Input({"type": "toggle-type-btn", "node_type": ALL}, "n_clicks"),

    State("hidden-store", "data"),

    prevent_initial_call=True
)
def hide_or_show(node1, node2, show_all, toggle_type_clicks, hidden_data):
    triggered = dash.ctx.triggered_id

    if hidden_data is None:
        hidden_data = {
            "cytoscape-1": [],
            "cytoscape-2": []
        }

    if triggered == "show-all-btn":
        hidden_data = {
            "cytoscape-1": [],
            "cytoscape-2": []
        }

    elif triggered == "cytoscape-1":
        if not node1:
            raise PreventUpdate

        node_id = node1["id"]
        hidden_list = hidden_data["cytoscape-1"]

        if node_id in hidden_list:
            hidden_list.remove(node_id)
        else:
            hidden_list.append(node_id)

    elif triggered == "cytoscape-2":
        if not node2:
            raise PreventUpdate

        node_id = node2["id"]
        hidden_list = hidden_data["cytoscape-2"]

        if node_id in hidden_list:
            hidden_list.remove(node_id)
        else:
            hidden_list.append(node_id)

    elif isinstance(triggered, dict) and triggered.get("type") == "toggle-type-btn":
        node_type = triggered.get("node_type")

        graph1_ids = get_node_ids_by_type(elements1_original, node_type)
        graph2_ids = get_node_ids_by_type(elements2_original, node_type)

        # Check if this node type is already fully hidden in both graphs
        graph1_hidden = all(
            node_id in hidden_data["cytoscape-1"]
            for node_id in graph1_ids
        )

        graph2_hidden = all(
            node_id in hidden_data["cytoscape-2"]
            for node_id in graph2_ids
        )

        type_is_hidden = graph1_hidden and graph2_hidden

        if type_is_hidden:
            # Show this node type again
            hidden_data["cytoscape-1"] = [
                node_id for node_id in hidden_data["cytoscape-1"]
                if node_id not in graph1_ids
            ]

            hidden_data["cytoscape-2"] = [
                node_id for node_id in hidden_data["cytoscape-2"]
                if node_id not in graph2_ids
            ]

        else:
            # Hide this node type
            for node_id in graph1_ids:
                if node_id not in hidden_data["cytoscape-1"]:
                    hidden_data["cytoscape-1"].append(node_id)

            for node_id in graph2_ids:
                if node_id not in hidden_data["cytoscape-2"]:
                    hidden_data["cytoscape-2"].append(node_id)

    else:
        raise PreventUpdate

    return (
        hidden_data,
        toggle_hidden_class(elements1_original, hidden_data["cytoscape-1"]),
        toggle_hidden_class(elements2_original, hidden_data["cytoscape-2"])
    )


if __name__ == "__main__":
    app.run(port=8070, debug=True)